# Fine-tune Gemma 4 E4B with Unsloth
## Ain-Bondhu: Bangla Worker Rights Assistant

This notebook fine-tunes **Gemma 4 E4B (Instruct)** on Bangla worker rights Q&A data.

### Setup
- Uses **Google Colab** (free T4 GPU, 16GB VRAM)
- **QLoRA** 4-bit quantization to fit in 16GB
- Saves to **Hugging Face Hub**
- Ready for deployment via vLLM on RunPod

In [ ]:
%%capture
!pip install unsloth
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [ ]:
import torch
from unsloth import FastLanguageModel, is_bfloat16_supported
from datasets import load_dataset
from transformers import TrainingArguments
from trl import SFTTrainer
import json, requests
import os

## 1. Load Model (4-bit QLoRA)

In [ ]:
# Load Gemma 4 E4B in 4-bit
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="google/gemma-4-E4B-it",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

print(f"Model loaded: {model.config._name_or_path}")
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 2. Load Training Dataset

In [ ]:
# Download training data from GitHub
DATA_URL = "https://raw.githubusercontent.com/hodini007/Ain-Bondhu/main/data/training/worker_rights_qa.json"
resp = requests.get(DATA_URL)
samples = resp.json()

# Format as instruction-output pairs
def format_sample(sample):
    return {
        "instruction": sample["instruction"],
        "output": sample["output"],
    }

formatted = [format_sample(s) for s in samples]
print(f"Loaded {len(formatted)} training samples")

In [ ]:
# Prompt template (Gemma 4 chat format)
prompt_template = """<bos><start_of_turn>user
{instruction}<end_of_turn>
<start_of_turn>model
{output}<end_of_turn></bos>"""

# Convert to dataset format
texts = []
for s in formatted:
    text = prompt_template.format(
        instruction=s["instruction"],
        output=s["output"],
    )
    texts.append(text)

print(f"Example:\n{texts[0][:200]}...")

In [ ]:
# Create dataset for trainer
dataset = load_dataset("json", data_files={"train": DATA_URL}, split="train")

def formatting_func(examples):
    texts = []
    for instr, out in zip(examples["instruction"], examples["output"]):
        text = prompt_template.format(instruction=instr, output=out)
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(formatting_func, batched=True)
print(f"Dataset ready: {len(dataset)} samples")

## 3. Fine-tune

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        report_to="none",
    ),
)

print("Starting training...")
trainer_stats = trainer.train()
print(f"Training complete! Loss: {trainer_stats.training_loss:.4f}")

In [ ]:
# Save adapter locally
model.save_pretrained("gemma4_bn_worker_lora")
tokenizer.save_pretrained("gemma4_bn_worker_lora")
print("Adapter saved to gemma4_bn_worker_lora/")

## 4. Push to Hugging Face Hub
Run this cell **after** setting your HF token.

In [ ]:
# Login to Hugging Face (run once)
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Push to Hub
HF_USER = "hodini007"  # CHANGE THIS
MODEL_NAME = "ain-bondhu-gemma4-lora"

model.push_to_hub(f"{HF_USER}/{MODEL_NAME}")
tokenizer.push_to_hub(f"{HF_USER}/{MODEL_NAME}")
print(f"Uploaded to https://huggingface.co/{HF_USER}/{MODEL_NAME}")

## 5. Test the Fine-tuned Model

In [ ]:
FastLanguageModel.for_inference(model)

test_prompt = """<bos><start_of_turn>user
আমার বেতন ৪ মাস বকেয়া, আমি কী করতে পারি?<end_of_turn>
<start_of_turn>model
"""

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.4, top_p=0.95)
response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(response)

In [ ]:
# Merge LoRA with base model for vLLM deployment
# This creates a single merged model file
merged_model = model.merge_and_unload()
merged_model.save_pretrained("gemma4_bn_worker_merged")
tokenizer.save_pretrained("gemma4_bn_worker_merged")
print("Merged model saved. Upload to RunPod for inference.")